In [9]:
import numpy as np
import scipy
import time
import math
import itertools
from scipy.stats import norm

from copy import deepcopy

from tomography import *


from NestedForLoop import get_iterator
from pathlib import Path
from scipy.linalg import sqrtm

import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import os
import glob

import pandas as pd

from scipy.optimize import least_squares

import fnmatch
from efficiencies import *
from optimization import Optimizer, function_fidelity_U4, FidelityResults, function_fidelity_Rz
from constants import *

from densitymatrix import DensityMatrix, apply_unitary_to_dm
from pathlib import Path
import fnmatch

In [10]:
working_dir = os.getcwd()
working_dir_data = r"C:\Users\QILIP6\Desktop\Multipartite Entanglement Experiment\Data\QST"
os.chdir(working_dir_data)
files="QST_GHZ_aqtime=150.0s_20250522114519"

######################################################################################################
#----- COUNTING THE FILES AND SAVING THEM IN AN ARRAY TO MAKES THE REST OF THE ANALYSIS EASIER -------
######################################################################################################

n_files=0
os.chdir(working_dir_data)

filenames = [i for i in glob.glob(files)]
filenames.sort(key=os.path.getmtime)

index_to_file = {}

for index, filename in enumerate(filenames):
    os.chdir(f"{working_dir_data}\\{filename}")
    filenames_aux=[i for i in glob.glob("counts*")]
    for index_second, filenames_aux_second in enumerate(filenames_aux):
        index_to_file[n_files] = f"{filename}\\{filenames_aux_second}"
        n_files+=1
os.chdir(working_dir)

print("Analyse Files: ", filenames_aux)
print(filenames)

Analyse Files:  ['counts']
['QST_GHZ_aqtime=150.0s_20250522114519']


In [11]:
######################################################################################################
#----- COUNTING THE FILES AND SAVING THEM IN AN ARRAY TO MAKES THE REST OF THE ANALYSIS EASIER -------
######################################################################################################

n_files=0
os.chdir(working_dir_data)

filenames = [i for i in glob.glob(files)]
filenames.sort(key=os.path.getmtime)

index_to_file = {}

for index, filename in enumerate(filenames):
    os.chdir(f"{working_dir_data}\\{filename}")
    filenames_aux=[i for i in glob.glob("counts*")]
    for index_second, filenames_aux_second in enumerate(filenames_aux):
        index_to_file[n_files] = f"{filename}\\{filenames_aux_second}"
        n_files+=1
os.chdir(working_dir)

print("Analyse Files: ", filenames_aux)
print(filenames)

Analyse Files:  ['counts']
['QST_GHZ_aqtime=150.0s_20250522114519']


In [12]:
#####################################################################
#---------------------- DEFINING PARAMS ----------------------------#
#####################################################################
os.chdir(working_dir)
qubit_number=4

## Defining the columns of the data file we want to use as data to reconstruct the density matrix (eg.: HH HV VH and VV basis)
BASIS_TO_CHANNEL={
    "HA": 1,
    "VA": 2,
    "HB": 3,
    "VB": 4,
    "HC": 5,
    "VC": 6,
    "HD": 7,
    "VD": 8,
    }

### Bell state ###
# eigenstates = [['HA','HB'],['HA','VB'],['VA','HB'],['VA','VB']]

### 4 qubits GHZ ###
eigenstates = [['HA','HB','HC','HD'],['HA','HB','HC','VD'],['HA','HB','VC','HD'],['HA','HB','VC','VD'],
               ['HA','VB','HC','HD'],['HA','VB','HC','VD'],['HA','VB','VC','HD'],['HA','VB','VC','VD'],
               ['VA','HB','HC','HD'],['VA','HB','HC','VD'],['VA','HB','VC','HD'],['VA','HB','VC','VD'],
               ['VA','VB','HC','HD'],['VA','VB','HC','VD'],['VA','VB','VC','HD'],['VA','VB','VC','VD']]

fold = four_fold=[[BASIS_TO_CHANNEL[eigenstates[i][j]] for j in range(qubit_number)] for i in range(len(eigenstates))]

In [13]:
import itertools

datafile_channels = fold.copy()

for clicks in fold:
    not_in_clicks = list(set(range(1, 2*qubit_number+1)) - set(clicks))
    not_in_clicks.sort()
    for rep in range(1, len(not_in_clicks)+1):
        for combo in itertools.combinations(not_in_clicks, rep):
            new_clicks = clicks + list(combo)
            new_clicks.sort()
            datafile_channels.append(new_clicks)
            
datafile_channels = np.array(list(set(map(tuple, datafile_channels))), dtype=object)
first_order = list(map(len, datafile_channels)) 
order = np.lexsort((datafile_channels, first_order))
datafile_channels = list(datafile_channels[order])
datafile_channels = [list(t) for t in datafile_channels]

In [14]:
# single_channels_offset = 8
coincidences_columns = []

# double_emission_columns = []

for i, iter in enumerate(eigenstates):
    proj = [BASIS_TO_CHANNEL[iter[m]] for m in range(qubit_number)]
    coincidences_columns.append(datafile_channels.index(proj))
#     double_emission_columns.append([datafile_channels.index(d) for d in datafile_channels if set(proj).issubset(set(d))])
#     double_emission_columns[-1].remove(coincidences_columns[-1])

column_start = np.min(coincidences_columns)# + single_channels_offset
column_stop = np.max(coincidences_columns) + 1# + single_channels_offset
print("Coincidences column_start:", column_start,"; column_stop: ", column_stop)

column_start_5_emissions = 2**qubit_number
column_stop_5_emissions = 2**qubit_number*qubit_number + column_start_5_emissions

column_start_6_emissions = column_stop_5_emissions
column_stop_6_emissions =  column_stop_5_emissions + 64*3

print("Coincidences column_start:", column_start_5_emissions,"; column_stop: ", column_stop_5_emissions)
print("Coincidences column_start:", column_start_6_emissions,"; column_stop: ", column_stop_6_emissions)

# column_start_2_emissions = np.min(double_emission_columns)# + single_channels_offset + 4
# column_stop_2_emissions = np.max(double_emission_columns)# + single_channels_offset + 4
# print("Double emission column_start:", column_start_2_emissions, "; column_stop: ", column_stop_2_emissions)

Coincidences column_start: 0 ; column_stop:  16
Coincidences column_start: 16 ; column_stop:  80
Coincidences column_start: 80 ; column_stop:  272


In [15]:
statetomo = []
state = []
state_file = []

xp_counts_corrected_with_eff=[]

#####################################################################
#---------------------- STATE TOMOGRAPHY ----------------------------
#####################################################################
for index in range(len(index_to_file)):
    os.chdir(f"{working_dir_data}\\{index_to_file[index]}\\")
    datafiles=[i for i in glob.glob("*")]
    
    ### Calculating the efficiencies of each detector
    efficiencies=get_channels_eff(datafiles, qubit_number, column_start, column_stop, os.getcwd())
    efficiencies_5_emissions=get_channels_eff(datafiles, qubit_number, column_start_5_emissions, column_stop_5_emissions, os.getcwd())
    efficiencies_6_emissions=get_channels_eff(datafiles, qubit_number, column_start_6_emissions, column_stop_6_emissions, os.getcwd())
   

    print("Channels efficiencies: ", efficiencies)
#    print("Channels efficiencies (5-fold): ", efficiencies_5_emissions)
#     print("Channels efficiencies (6-fold): ", efficiencies_6_emissions)

    ### Opening the data files and writing the data in counts_aux array
    counts_aux=set_raw_counts(datafiles, qubit_number, column_start, column_stop, os.getcwd())
    xp_counts=np.array(np.transpose(counts_aux))
    total_per_basis=np.sum(xp_counts, axis=1)
 
    counts_aux_5_emissions=set_raw_counts_double_emissions(datafiles, qubit_number, column_start_5_emissions, column_stop_5_emissions, os.getcwd())
    xp_counts_5_emissions=np.array(np.transpose(counts_aux_5_emissions))

    counts_aux_6_emissions=set_raw_counts_double_emissions(datafiles, qubit_number, column_start_6_emissions, column_stop_6_emissions, os.getcwd())
    xp_counts_6_emissions=np.array(np.transpose(counts_aux_6_emissions))

    statetomo.append(LRETomography(int(qubit_number), xp_counts, xp_counts_5_emissions,xp_counts_6_emissions))
    statetomo[-1].run(correct_eff=efficiencies,correct_double_emission_eff=efficiencies_5_emissions,GHZ = True)
    xp_counts_corrected_with_eff.append(statetomo[-1].xp_counts)
        
    state.append(statetomo[-1])
    state_file.append(index_to_file[index])

Channels efficiencies:  [0.7037037  0.87742504 0.60582011 0.77336861 0.7292769  0.99118166
 0.64109347 0.7962963  0.6984127  0.85978836 0.64814815 0.7319224
 0.78747795 1.         0.63580247 0.85537919]


c:\Users\QILIP6\Desktop\Multipartite Entanglement Experiment\Analysis_code\Tomography\projectorcounts.py:133: RuntimeWarning: divide by zero encountered in divide
  self.counts_array_2_emissions[w] = self.counts_array_2_emissions[w]/double_emission_eff.astype(float)
c:\Users\QILIP6\Desktop\Multipartite Entanglement Experiment\Analysis_code\Tomography\projectorcounts.py:133: RuntimeWarning: invalid value encountered in divide
  self.counts_array_2_emissions[w] = self.counts_array_2_emissions[w]/double_emission_eff.astype(float)


In [ ]:
import numpy as np
from scipy.optimize import minimize
from functools import reduce
from itertools import product
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import matplotlib.colors as mcolors
import seaborn as sns
import time
from joblib import Parallel, delayed

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
# --- MLE Settings ---
OPTIMIZER_TOLERANCE = 1e-25 # Machine precision tolerance
GRADIENT_TOLERANCE = 1e-25 
MAX_ITERATIONS = 150000000      
N_RESTARTS = 1             # Number of solver restarts

# --- Monte Carlo Settings ---
MC_SAMPLES = 100      
MC_CORES = -1               # Use all cores
MC_SHOT_NOISE = True        
MC_CALIB_NOISE = True       
WP_SIGMA = 0.5 * np.pi / 180 

# --- Physics Constants ---
HWP_SETTINGS = {"x": np.pi/8, "y": 0, "z": 0}
QWP_SETTINGS = {"x": np.pi/2, "y": 3*np.pi/4, "z": np.pi/2}

# ==============================================================================
# 2. PHYSICS 
# ==============================================================================

def get_outcome_iterator(dim_base, n_qubits):
    iterator = []
    for r in range(dim_base**n_qubits):
        r_in_base = np.base_repr(r, dim_base)
        outcome_list = [0] * n_qubits
        for c in range(len(r_in_base)):
            outcome_list[n_qubits - len(r_in_base) + c] = int(r_in_base[c])
        iterator.append(np.array(outcome_list))
    return iterator

def wp_rotation(theta, retardance):
    phase_factor = np.exp(-1j * retardance / 2)
    c, s = np.cos(theta), np.sin(theta)
    return phase_factor * np.array([
        [c**2 + np.exp(1j * retardance)*s**2,       (1 - np.exp(1j * retardance))*c*s],
        [(1 - np.exp(1j * retardance))*c*s,       s**2 + np.exp(1j * retardance)*c**2]
    ])

def get_single_qubit_projector(hwp, qwp, bit):
    U_tot = wp_rotation(qwp, np.pi/2) @ wp_rotation(hwp, np.pi)
    lab_basis = np.array([1, 0], dtype=complex) if bit == 0 else np.array([0, 1], dtype=complex)
    return U_tot.conj().T @ lab_basis

def generate_povms(n_qubits, offsets=None):
    if offsets is None: offsets = [(0,0)] * n_qubits
    
    # Map integers from the iterator to dictionary keys
    # 0 -> 'x', 1 -> 'y', 2 -> 'z'
    int_to_basis = {0: 'x', 1: 'y', 2: 'z'} 
    
    # 1. Use get_outcome_iterator for BASES (Base 3)
    # Note: This builds the whole list in memory!
    basis_indices_list = get_outcome_iterator(3, n_qubits)
    
    # 2. Use get_outcome_iterator for OUTCOMES (Base 2)
    outcome_iterator = get_outcome_iterator(2, n_qubits)
    
    povm_vectors = []
    
    # Loop over the integer arrays (e.g., [0, 2] instead of "xz")
    for basis_indices in basis_indices_list:
        for outcome_tuple in outcome_iterator:
            qubit_projectors = []
            for q in range(n_qubits):
                # Map Integer to Char
                axis_int = basis_indices[q]
                axis_char = int_to_basis[axis_int] 
                
                bit = outcome_tuple[q]
                
                # Use the mapped char to get angles
                h = HWP_SETTINGS[axis_char] + offsets[q][0]
                q_ang = QWP_SETTINGS[axis_char] + offsets[q][1]
                
                qubit_projectors.append(get_single_qubit_projector(h, q_ang, bit))
            povm_vectors.append(reduce(np.kron, qubit_projectors))
            
    return np.array(povm_vectors)

# ==============================================================================
# 3. MLE 
# ==============================================================================

def params_to_rho(x, dim):
    size = dim * dim
    t_real = x[:size].reshape((dim, dim))
    t_imag = x[size:].reshape((dim, dim))
    T = np.tril(t_real) + 1j * np.tril(t_imag, k=-1)
    rho_u = T @ T.conj().T
    tr = np.real(np.trace(rho_u))
    return rho_u / (tr if tr > 1e-15 else 1.0)

def analyze_tomography_mle(counts_matrix, povms, n_qubits, verbose=True,solver = 'L-BFGS-B'):
    rng_state = np.random.RandomState()
    dim = 2**n_qubits
    
    # 1. Flatten Data
    data_flat = []
    povm_indices = []
    idx_counter = 0
    for b_idx in range(counts_matrix.shape[1]):
        col = counts_matrix[:, b_idx]
        if np.sum(col) > 0:
            data_flat.extend(col)
            povm_indices.extend(range(idx_counter, idx_counter + dim))
        idx_counter += dim     
    data_flat = np.array(data_flat)
    active_povms = povms[povm_indices]
    
    # 2. NLL
    def nll_func(x):
        rho = params_to_rho(x, dim)
        temp = rho @ active_povms.T
        probs = np.real(np.sum(active_povms.conj().T * temp, axis=0))
        probs = np.maximum(probs, 1e-15)
        return -np.sum(data_flat * np.log(probs))

    best_fun = np.inf
    best_rho = None
    
    # 3. Optimization Loop
    for i in range(N_RESTARTS):
        if verbose: print(f"\n--- [MLE] Restart {i+1}/{N_RESTARTS} ---")
        
        if i == 0:
            x0 = np.concatenate([np.eye(dim).flatten(), np.zeros(dim*dim)]) 
            x0 += rng_state.normal(0, 0.05, 2*dim*dim)
        else:
            x0 = rng_state.normal(0, 0.2, 2*dim*dim)
            
        res = minimize(nll_func, x0, method=solver, 
                       options={
                           'maxiter': MAX_ITERATIONS, 
                           'ftol': OPTIMIZER_TOLERANCE, 
                           'gtol': GRADIENT_TOLERANCE,
                           'maxfun' : 10000000,
                           'disp': verbose,
                           
                       })
        
        if res.fun < best_fun:
            best_fun = res.fun
            best_rho = params_to_rho(res.x, dim)
        if verbose: print(f"    > New Best NLL: {res}")
            
    return best_rho

# ==============================================================================
# 4. UNITARY CORRECTION & METRICS
# ==============================================================================

def get_fidelity(rho1, rho2):
    vals, vecs = np.linalg.eigh(rho1)
    sq1 = vecs @ np.diag(np.maximum(vals,0)**0.5) @ vecs.conj().T
    term = sq1 @ rho2 @ sq1
    vals_t, vecs_t = np.linalg.eigh(term)
    return np.real(np.trace(vecs_t @ np.diag(np.maximum(vals_t,0)**0.5) @ vecs_t.conj().T))**2

def u3_matrix(t, p, l):
    c, s = np.cos(t/2), np.sin(t/2)
    return np.array([[c, -np.exp(1j*l)*s],[np.exp(1j*p)*s, np.exp(1j*(p+l))*c]])

def Unitary(angle, u):
    a, b, y = angle[0], angle[1], angle[2]
    
    f = (1/2)*(-np.cos(2*(a-b))-np.cos(2*(b-y))) - np.real(u[0][0])
    g = (1/2)*(np.sin(2*(a - b)) + np.sin(2*(b - y))) - np.real(u[0][1])
    h = (1/2)*(-np.sin(2*(a - b)) - np.sin(2*(b - y))) - np.real(u[1][0])
    v = (1/2)*(-np.cos(2*(a-b))- np.cos(2*(b - y))) - np.real(u[1][1])

    K = (1/2)*(-np.cos(2*b) + np.cos(2*(a - b + y))) - np.imag(u[0][0])
    m = (1/2)*(-np.sin(2*b) + np.sin(2*(a - b + y))) - np.imag(u[0][1])
    z = (1/2)*(-np.sin(2*b) + np.sin(2*(a - b + y)))- np.imag(u[1][0])
    e = (1/2)*(np.cos(2*b) - np.cos(2*(a - b + y))) - np.imag(u[1][1])

    return (f,g,h,v,K,m,z,e)

def solving(angle, u):
    # Uses least_squares to find QWP-HWP-QWP angles
    result = least_squares(Unitary, angle, method='trf', args=[u], max_nfev=1000000)
    QWP1 = result.x[0]
    HWP1 = result.x[1]
    QWP2 = result.x[2]
    return [QWP2*180/np.pi, HWP1*180/np.pi, QWP1*180/np.pi]

def optimize_unitary_correction(rho_exp, rho_target, n_qubits):
    print(f"   > Optimization: Aligning reference frame...")
    def cost(params):
        us = [u3_matrix(params[3*i], params[3*i+1], params[3*i+2]) for i in range(n_qubits)]
        U = reduce(np.kron, us)
        rho_prime = U @ rho_exp @ U.conj().T
        return -np.real(np.trace(rho_prime @ rho_target))
    
    res = minimize(cost, np.zeros(3*n_qubits), method='SLSQP', options={'ftol':1e-12})
    
    us_components = [u3_matrix(res.x[3*i], res.x[3*i+1], res.x[3*i+2]) for i in range(n_qubits)]
    return -res.fun, reduce(np.kron, us_components), us_components

# ==============================================================================
# 5. PARALLEL MONTE CARLO
# ==============================================================================

def _mc_worker(seed, counts_shape, shots_total, rho_truth, rho_target, U_corr, n_qubits):
    rng = np.random.RandomState(seed)
    dim = 2**n_qubits
    dim_out, dim_base = counts_shape
    
    offsets = []
    if MC_CALIB_NOISE:
        for _ in range(n_qubits): offsets.append((rng.normal(0, WP_SIGMA), rng.normal(0, WP_SIGMA)))
    
    mc_povms = generate_povms(n_qubits, offsets)
    temp = rho_truth @ mc_povms.T
    probs = np.real(np.sum(mc_povms.conj().T * temp, axis=0))
    probs = np.maximum(probs, 0)
    
    sim_counts = np.zeros(counts_shape)
    idx = 0
    for b in range(dim_base):
        p_basis = probs[idx:idx+dim]
        if np.sum(p_basis) > 0: p_basis /= np.sum(p_basis)
        else: p_basis = np.ones(dim)/dim
        if MC_SHOT_NOISE: sim_counts[:, b] = rng.multinomial(shots_total, p_basis)
        else: sim_counts[:, b] = p_basis * shots_total
        idx += dim
    
    blind_povms = generate_povms(n_qubits, None)
    rho_sim = analyze_tomography_mle(sim_counts, blind_povms, n_qubits, verbose=False,solver='L-BFGS-B')
    if U_corr is not None: rho_sim = U_corr @ rho_sim @ U_corr.conj().T
        
    return get_fidelity(rho_sim, rho_target)

def run_monte_carlo_parallel(rho_master, counts_shape, shots_total, rho_target, U_corr, n_qubits):
    print(f"   > Monte Carlo: Launching {MC_SAMPLES} parallel jobs...")
    global_rng = np.random.RandomState() 
    seeds = global_rng.randint(0, 1e9, size=MC_SAMPLES)
    results = Parallel(n_jobs=MC_CORES, verbose=0)(
        delayed(_mc_worker)(seed, counts_shape, shots_total, rho_master, rho_target, U_corr, n_qubits) 
        for seed in seeds
    )
    return np.mean(results), np.std(results), results

# ==============================================================================
# 6. 3D VISUALIZATION
# ==============================================================================

def plot_density_matrix_3d_custom(rho, n_qubits, cbar_real=True, cbar_im=True, save_pdf=None):
    real_density_matrix = rho.real
    imag_density_matrix = rho.imag
    
    # 1. Labels & Grid
    HV_label = {0: "H", 1: "V"}
    def get_iterator_simple(base, n):
        it = []
        for i in range(base**n):
            binary = np.binary_repr(i, width=n)
            it.append([int(b) for b in binary])
        return it
    HV_iterator = get_iterator_simple(2, n_qubits)
    axes_labels = []
    for k in range(np.shape(rho)[0]):
        axes_labels.append("".join([HV_label[bit] for bit in HV_iterator[k]]))

    nrows, ncols = rho.shape
    x, y = np.arange(ncols), np.arange(nrows)
    X, Y = np.meshgrid(x, y)
    Z_real, Z_imag = real_density_matrix.flatten(), imag_density_matrix.flatten()
    heights_real, heights_imag = np.abs(Z_real), np.abs(Z_imag)
    width = depth = 0.7

    # 2. GLOBAL SCALE CALCULATION (Crucial for consistent Z-axis)
    max_height_global = max(np.max(heights_real), np.max(heights_imag))
    if max_height_global < 1e-9: max_height_global = 1.0 # Safety
    # Add 10% headroom
    z_limit = max_height_global * 1.1

    # 3. Color Logic
    cmap = plt.get_cmap("coolwarm")
    def get_colors_symmetric(values):
        max_val = np.max(np.abs(values))
        if max_val < 1e-9: max_val = 1.0
        norm = mcolors.Normalize(vmin=0, vmax=max_val)
        return cmap(norm(values)), norm

    # --- REAL PART ---
    fig1 = plt.figure(figsize=(9, 6))
    ax1 = fig1.add_subplot(111, projection="3d")
    colors_real, norm_real = get_colors_symmetric(Z_real)
    
    ax1.bar3d(X.ravel(), Y.ravel(), np.zeros_like(Z_real), width, depth, heights_real,
              shade=True, color=colors_real, alpha=0.85, edgecolor='black', linewidth=0.1)
    
    ax1.set_xticks(np.arange(ncols)+0.5); ax1.set_yticks(np.arange(nrows)+0.5)
    ax1.set_xticklabels(axes_labels, rotation=45); ax1.set_yticklabels(axes_labels, rotation=-60)
    ax1.set_title("Real Part $\mathcal{R}(\\rho)$", fontsize=14)
    ax1.set_zlim(0, z_limit) # <--- Apply Global Limit
    
    if cbar_real: fig1.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm_real), ax=ax1, shrink=0.7).set_label("Amplitude")
    ax1.view_init(elev=25, azim=-45)
    if save_pdf: plt.savefig(save_pdf+"_real.pdf")
    plt.show()

    # --- IMAGINARY PART ---
    fig2 = plt.figure(figsize=(9, 6))
    ax2 = fig2.add_subplot(111, projection="3d")
    colors_imag, norm_imag = get_colors_symmetric(Z_imag)
    
    ax2.bar3d(X.ravel(), Y.ravel(), np.zeros_like(Z_imag), width, depth, heights_imag,
              shade=True, color=colors_imag, alpha=0.85, edgecolor='black', linewidth=0.1)
    
    ax2.set_xticks(np.arange(ncols)+0.5); ax2.set_yticks(np.arange(nrows)+0.5)
    ax2.set_xticklabels(axes_labels, rotation=45); ax2.set_yticklabels(axes_labels, rotation=-60)
    ax2.set_title("Imaginary Part $\mathcal{I}(\\rho)$", fontsize=14)
    ax2.set_zlim(0, z_limit) # <--- Apply Global Limit
    
    if cbar_im: fig2.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm_imag), ax=ax2, shrink=0.7).set_label("Amplitude")
    ax2.view_init(elev=25, azim=-45)
    if save_pdf: plt.savefig(save_pdf+"_imag.pdf")
    plt.show()

# ==============================================================================
# 7. MAIN EXECUTION
# ==============================================================================

if __name__ == "__main__":
    
    if 'statetomo' in locals(): raw = statetomo[-1].xp_counts.counts_array
    elif 'ata' in locals(): raw = ata
    else: raise NameError
    if raw.shape[0] < raw.shape[1]: counts = raw
    else: counts = raw.T
    N_QUBITS = int(np.log2(counts.shape[0]))

    print(f"\n{'='*60}")
    print(f" QUANTUM TOMOGRAPHY (N={N_QUBITS})")
    print(f"{'='*60}")

    print(f" Performing MLE Reconstruction...")
    t0 = time.time()
    ideal_povms = generate_povms(N_QUBITS)
    rho_mle = analyze_tomography_mle(counts, ideal_povms, N_QUBITS, verbose=True)
    print(f"      Done. Time: {time.time()-t0:.3f}s")

    d = 2**N_QUBITS
    t_vec = np.zeros((d, 1), dtype=complex)
    t_vec[0] = 1; t_vec[-1] = 1; t_vec /= np.sqrt(2)
    rho_target = t_vec @ t_vec.conj().T

    print(f"Optimizing Reference Frame...")
    fid_raw = get_fidelity(rho_mle, rho_target)
    fid_opt, U_corr,us_opts = optimize_unitary_correction(rho_mle, rho_target, N_QUBITS)
    rho_final = U_corr @ rho_mle @ U_corr.conj().T

    # --- IMMEDIATE RESULTS DISPLAY ---
    purity = np.real(np.trace(rho_final @ rho_final))
    trace_val = np.real(np.trace(rho_final))
    eigvals = np.linalg.eigvalsh(rho_final)

    print(f"\n{'='*60}")
    print(f" INTERMEDIATE ANALYSIS REPORT (N={N_QUBITS})")
    print(f"{'='*60}")
    print(f"  > Raw Fidelity:           {fid_raw:.4f}")
    print(f"  > Corrected Fidelity:     {fid_opt:.4f}")
    print(f"  > Purity:                 {purity:.4f}")
    print(f"  > Trace:                  {trace_val:.6f}")
    print(f"  > Min Eigenvalue:         {np.min(eigvals):.6e}")
    
    print(f"{'='*60}\n")

    # --- PRINT ANGLES ---
    print(f"\n{'='*60}")
    print(f"Physical Waveplate Corrections:")
    print(f"{'='*60}")
    qubit_names = ["arya", "bran", "cersei", "dany"] # Map index to name
    
    for i, u_local in enumerate(us_opts):
        angles = solving([0,0,0], u_local)
        name = qubit_names[i] if i < len(qubit_names) else f"qubit_{i}"
        formatted_angles = [float(f"{a:.5f}") for a in angles]
        print(f"{name}.set_sample_angles({formatted_angles})\n")

    # --- NOW RUN MONTE CARLO ---
    print(f"Estimating Errors (Monte Carlo)...")
    shots_est = int(np.mean(np.sum(counts, axis=0)))
    mu, sigma, all_fids = run_monte_carlo_parallel(rho_mle, counts.shape, shots_est, rho_target, U_corr, N_QUBITS)

    # --- FINAL REPORT ---
    print(f"\n{'='*60}")
    print(f" FINAL REPORT")
    print(f"{'='*60}")
    print(f"  > Final Fidelity:         {fid_opt:.4f} ± {sigma:.4f}")
    print(f"{'='*60}")

    plot_density_matrix_3d_custom(rho_final, N_QUBITS)
    
    plt.figure(figsize=(6, 4))
    sns.histplot(all_fids, kde=True, color="green", bins=15, alpha=0.6)
    plt.axvline(fid_opt, color='black', linestyle='--', label="Measured")
    plt.title(f"Error Analysis ({MC_SAMPLES} samples)")
    plt.xlabel("Fidelity"); plt.legend()
    plt.tight_layout(); plt.show()



<>:311: SyntaxWarning: invalid escape sequence '\m'
<>:329: SyntaxWarning: invalid escape sequence '\m'
<>:311: SyntaxWarning: invalid escape sequence '\m'
<>:329: SyntaxWarning: invalid escape sequence '\m'
C:\Users\QILIP6\AppData\Local\Temp\ipykernel_3712\1205296112.py:311: SyntaxWarning: invalid escape sequence '\m'
  ax1.set_title("Real Part $\mathcal{R}(\\rho)$", fontsize=14)
C:\Users\QILIP6\AppData\Local\Temp\ipykernel_3712\1205296112.py:329: SyntaxWarning: invalid escape sequence '\m'
  ax2.set_title("Imaginary Part $\mathcal{I}(\\rho)$", fontsize=14)



 QUANTUM TOMOGRAPHY (N=4)
 Performing MLE Reconstruction...

--- [MLE] Restart 1/1 ---
    > New Best NLL:   message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 324910.80832884106
        x: [ 1.013e+01 -5.007e-02 ... -5.550e-04  2.117e-02]
      nit: 391
      jac: [ 1.164e-02  0.000e+00 ...  0.000e+00  0.000e+00]
     nfev: 229311
     njev: 447
 hess_inv: <512x512 LbfgsInvHessProduct with dtype=float64>
      Done. Time: 57.510s
Optimizing Reference Frame...
   > Optimization: Aligning reference frame...

 INTERMEDIATE ANALYSIS REPORT (N=4)
  > Raw Fidelity:           0.0591
  > Corrected Fidelity:     0.9627
  > Purity:                 0.9287
  > Trace:                  1.000000
  > Min Eigenvalue:         7.896162e-18


Physical Waveplate Corrections:
arya.set_sample_angles([25.0518, -36.78581, -107.7482])

bran.set_sample_angles([-28.57565, 36.15612, 111.52879])

cersei.set_sample_angles([39.32354, -3.73167, -75.90027])

dany.set_samp